# DPM Engine · Metadata Load

**Purpose.** Loads the parameterisation files (one Excel per framework) and the issue catalogue into the Lakehouse, keeping the full history of versions (SCD Type 2.2).

- Reads every `.xlsx` file of the configuration folder `Files/NB-Framework/dpm/config`. Each file must contain the tabs Datapoints, Filters, Catalog and Sources.
- Publishes `dp_lgl_reg_dpm_param_datapoint`, `_param_fltr`, `_param_ctlg` and `_param_src_tbl`, and the issue catalogue `_param_iss_ctlg` from `Files/NB-Framework/dpm/DPM_Issue_Catalog.xlsx`.
- The load is a full snapshot of the folder: new or changed rows create a new version, unchanged rows keep their version, and rows that disappear are closed with a logical deletion (`row_present_flg = 0`).
- All tables and files of the same load share one version timestamp, which identifies the resulting parameterisation snapshot.
- Every file is read and checked before any table is published; a missing tab or mandatory Sources column stops the load without changing the current parameterisation.

**Coexistence with the DataLoader load (2026-11-20).** The parameterisation tables can also be loaded through the DataLoader (`DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD`). Those rows have no `src_row_nr`: this notebook leaves them as they are, and the full-snapshot deletion only applies to the rows it loaded itself. Each framework must be loaded through one of the two ways only: a framework present in both would have its datapoints twice.

**Next step:** run `DE_ENR_NB_DPM_Metadata_Validation` for the affected frameworks.

**Location:** workspace *DP FET Legal and Regulatory Reporting*, folder `01_Processes/DPM`, default Lakehouse `DP_FET_LH_LegalAndRegulatoryReporting`. All engine tables carry the prefix `dp_lgl_reg_dpm_`.


## 1. Parameters


In [1]:
%%configure -f
{
    "driverMemory": "7g",
    "driverCores": 2,
    "executorMemory": "7g",
    "executorCores": 2,
    "conf": {
        "spark.native.enabled": "true",
        "spark.dynamicAllocation.maxExecutors": "1"
    },
    "sessionTimeoutInSeconds": 1200
}

StatementMeta(, c5e6a7b0-a131-4635-85b4-3f25991aaad8, -1, Finished, Available, Finished, True)

<div style="font-family:monospace;font-size:13px">
    <p>
        <span style="color:red;font-weight:bold">Warning:</span> When you want to configure the driverMemory, driverCores, executorCores,
        executorMemory, or numExecutors, check the workspace settings and default pool node details from workspace settings -> Default pool
        for workspace -> Pool details, you can use %%configure set customized compute configurations that within the range of Pool level size.
    </p>
    <p>
        Notice that if the session configuration is different from the live session (8 vCores, 56GB memory, 2 executors), the session
        start will fall back to the on-demand session which will take about 3~5 mins to spin up.
    </p>
    <p>The Executor/Driver size buckets on Fabric are: </p>
    <ul>
        <li>A small size bucket: 4 vCores/28 GB memory</li>
        <li>A medium size bucket: 8 vCores/56 GB memory</li>
        <li>A large size bucket: 16 vCores/112 GB memory</li>
        <li>A xlarge size bucket: 32 vCores/224 GB memory</li>
        <li>A xxlarge size bucket: 64 vCores/400 GB memory</li>
    </ul>
    <p>
        We recommend avoiding using Driver or Executor vCore/Memory counts that are not in the size bucket specified above, so be sure
        to specify the <b>Driver and Executor</b> vCore/Memory values in relation to the above size buckets (4vCores/28GB, 8vCores/56GB,
        16vCores/112GB, 32vCores/224GB, or 64vCores/400GB), moreover, the executor size and driver size should be the same.
    </p>
</div>

In [2]:
# Configuration folder with the parameterisation files (one Excel per framework).
metadata_input_path = "/lakehouse/default/Files/NB-Framework/dpm/config"
# Mandatory tabs of each parameterisation file.
required_sheets = ["Datapoints", "Filters", "Catalog", "Sources"]
fail_on_empty_folder = True

# Direct Spark/Delta SCD settings.
fail_on_schema_drift = True


# Version timestamp of the load. Empty = current UTC time, shared by the whole load.
# It becomes the functional version boundary used by ref_start_dt/ref_end_dt.
load_version_timestamp = ""

# Issue catalogue. Kept outside the configuration folder so that it is not mistaken
# for a parameterisation file.
load_issue_catalog = True
issue_catalog_input_path = "/lakehouse/default/Files/NB-Framework/dpm/DPM_Issue_Catalog.xlsx"
issue_catalog_sheet = "IssueCatalog"
issue_catalog_table = "dp_lgl_reg_dpm_param_iss_ctlg"


StatementMeta(, c5e6a7b0-a131-4635-85b4-3f25991aaad8, 5, Finished, Available, Finished, False)

## 2. Novo Banco SCD 2.2 / WorkerFactory helpers


In [3]:
import re
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.ms.autotune.enabled", "true")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.session.timeZone", "UTC")

DPM_VERSION_COLUMN = "dpm_vrs_dt"
DPM_INGESTION_COLUMN = "dpm_ingestion_dt"
DPM_PRESENT_COLUMN = "row_present_flg"

SCD_CONTROL_PREFIXES = ("sk_", "ref_", "ctrl_")
SAFE_NAME_PATTERN = re.compile(r"[^A-Za-z0-9_]+")


def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def resolve_load_timestamp(value):
    """Use one timestamp for the complete load. Empty means current UTC time."""
    text = str(value or "").strip()
    if not text:
        return utc_now()

    parsed = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if parsed.tzinfo is not None:
        parsed = parsed.astimezone(timezone.utc).replace(tzinfo=None)
    return parsed


def safe_name(value, max_length=80):
    normalized = SAFE_NAME_PATTERN.sub("_", str(value)).strip("_") or "DPM"
    if normalized[0].isdigit():
        normalized = "DPM_" + normalized
    return normalized[:max_length]


def surrogate_key_column(table_name):
    base_name = str(table_name).split(".")[-1]
    return f"sk_{safe_name(base_name, 100).lower()}"


def is_scd_control_column(column_name):
    return str(column_name).lower().startswith(SCD_CONTROL_PREFIXES)


def assert_unique_join_keys(frame, join_keys, table_name):
    missing = [c for c in join_keys if c not in frame.columns]
    if missing:
        raise ValueError(
            f"SCD join key column(s) {missing} do not exist in incoming table "
            f"{table_name!r}."
        )

    duplicate = (
        frame.groupBy(*[F.col(c) for c in join_keys])
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .collect()
    )
    if duplicate:
        raise ValueError(
            f"Incoming snapshot for {table_name!r} contains duplicate SCD join keys "
            f"{join_keys}. SCD Type 2.2 requires one incoming row per logical key."
        )


def assert_scd_target_compatible(table_name, incoming_columns):
    """Fail fast if an existing Delta target has an incompatible payload schema."""
    if not spark.catalog.tableExists(table_name):
        return

    existing = spark.table(table_name)
    existing_lower = {c.lower() for c in existing.columns}

    required_controls = {
        surrogate_key_column(table_name).lower(),
        "ref_start_dt",
        "ref_end_dt",
        "ref_last_dt",
        "ctrl_start_dt",
        "ctrl_end_dt",
        "ctrl_updt_dt",
        "ctrl_state",
    }
    missing_controls = sorted(required_controls - existing_lower)
    if missing_controls:
        raise RuntimeError(
            f"Target table {table_name!r} exists but is not compatible with this "
            f"Spark SCD 2.2 implementation. Missing control columns: {missing_controls}"
        )

    existing_payload = {
        c.lower()
        for c in existing.columns
        if not is_scd_control_column(c)
    }
    incoming_payload = {c.lower() for c in incoming_columns}

    if existing_payload != incoming_payload:
        missing_in_input = sorted(existing_payload - incoming_payload)
        new_in_input = sorted(incoming_payload - existing_payload)
        raise RuntimeError(
            f"Schema drift detected for {table_name!r}. "
            f"Columns missing from incoming snapshot: {missing_in_input}; "
            f"new incoming columns: {new_in_input}."
        )


def build_full_snapshot_with_tombstones(
    table_name,
    incoming,
    join_keys,
    load_timestamp,
    snapshot_mode="FULL",
):
    """
    Preserve full-snapshot delete semantics.

    In FULL mode, keys that are active in Delta but absent from the new workbook
    snapshot are converted into incoming tombstone rows. The normal SCD merge will
    then close the previous active version and insert a new version with
    row_present_flg = 0.
    """
    mode = str(snapshot_mode or "FULL").strip().upper()
    if mode not in {"FULL", "UPSERT"}:
        raise ValueError(
            f"Unsupported snapshot_mode {snapshot_mode!r}; use FULL or UPSERT."
        )

    assert_scd_target_compatible(table_name, incoming.columns)
    assert_unique_join_keys(incoming, join_keys, table_name)

    if mode == "UPSERT" or not spark.catalog.tableExists(table_name):
        return incoming, 0

    current = spark.table(table_name).filter(F.col("ctrl_state") == F.lit(1))
    if DPM_PRESENT_COLUMN in current.columns:
        current = current.filter(F.col(DPM_PRESENT_COLUMN) == F.lit(1))
    # Rows loaded through the DataLoader (DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD) have no
    # src_row_nr: this notebook neither replaces nor deletes them.
    if "src_row_nr" in join_keys:
        current = current.filter(F.col("src_row_nr").isNotNull())

    incoming_keys = incoming.select(*join_keys).dropDuplicates()
    missing_current = current.join(incoming_keys, join_keys, "left_anti")

    tombstone_count = missing_current.count()
    if tombstone_count == 0:
        return incoming, 0

    tombstones = missing_current.select(*incoming.columns)
    tombstones = (
        tombstones
        .withColumn(DPM_PRESENT_COLUMN, F.lit(0).cast("int"))
        .withColumn(DPM_VERSION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
        .withColumn(DPM_INGESTION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
    )

    if "load_dt" in tombstones.columns:
        tombstones = tombstones.withColumn(
            "load_dt",
            F.lit(load_timestamp).cast("timestamp"),
        )

    return incoming.unionByName(tombstones), tombstone_count


def _quoted_join_condition(left_alias, right_alias, join_keys):
    return " AND ".join(
        f"{left_alias}.`{c}` <=> {right_alias}.`{c}`"
        for c in join_keys
    )


def _payload_hash(frame, columns):
    """
    Null-safe deterministic row hash.

    sha2(to_json(struct(...))) avoids ambiguous string concatenation and preserves
    column boundaries.
    """
    return F.sha2(
        F.to_json(
            F.struct(*[
                F.col(c).alias(c)
                for c in columns
            ])
        ),
        256,
    )


def _with_new_scd_control_columns(frame, table_name, base_sk):
    """
    Add a new SCD version's technical columns.

    Surrogate keys are allocated sequentially above the current maximum. Metadata
    volumes are small, so a global row_number window is acceptable and predictable.
    """
    sk_column = surrogate_key_column(table_name)

    order_columns = [
        F.col(c).asc_nulls_first()
        for c in frame.columns
        if c not in {DPM_INGESTION_COLUMN}
    ]
    if not order_columns:
        order_columns = [F.lit(1)]

    sequence_window = Window.orderBy(*order_columns)

    return (
        frame
        .withColumn(
            sk_column,
            (F.lit(int(base_sk)) + F.row_number().over(sequence_window)).cast("long"),
        )
        .withColumn("ref_start_dt", F.col(DPM_VERSION_COLUMN).cast("timestamp"))
        .withColumn("ref_end_dt", F.lit(None).cast("timestamp"))
        .withColumn("ref_last_dt", F.col(DPM_VERSION_COLUMN).cast("timestamp"))
        .withColumn("ctrl_start_dt", F.col(DPM_INGESTION_COLUMN).cast("timestamp"))
        .withColumn("ctrl_end_dt", F.lit(None).cast("timestamp"))
        .withColumn("ctrl_updt_dt", F.col(DPM_INGESTION_COLUMN).cast("timestamp"))
        .withColumn("ctrl_state", F.lit(1).cast("int"))
    )


def create_scd22_table(table_name, incoming):
    """Create a new Delta SCD table and insert the complete first snapshot."""
    assert_unique_join_keys(
        incoming,
        [c for c in incoming.columns if False],
        table_name,
    ) if False else None

    first_versions = _with_new_scd_control_columns(
        incoming,
        table_name=table_name,
        base_sk=0,
    )

    (
        first_versions.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(table_name)
    )

    return incoming.count()


def spark_scd22_upsert(
    *,
    frame,
    target_table,
    join_keys,
    ignore_columns_to_hash=None,
):
    """
    Direct Spark/Delta SCD 2.2 publication.

    Behaviour:
      * missing table -> create and insert the first snapshot;
      * new key -> insert active version;
      * changed key -> close current active version + insert new active version;
      * unchanged key -> keep the version and advance ref_last_dt / ctrl_updt_dt;
      * tombstone -> treated as a normal changed version because row_present_flg
        participates in the comparison hash.

    `ignore_columns_to_hash` are carried into new versions but do not themselves
    trigger a new SCD version.
    """
    ignore_columns_to_hash = set(ignore_columns_to_hash or [])

    assert_unique_join_keys(frame, join_keys, target_table)
    assert_scd_target_compatible(target_table, frame.columns)

    if not spark.catalog.tableExists(target_table):
        inserted = create_scd22_table(target_table, frame)
        return {
            "created": True,
            "inserted_versions": inserted,
            "closed_versions": 0,
            "unchanged_rows": 0,
        }

    target = spark.table(target_table)
    sk_column = surrogate_key_column(target_table)

    # The two SCD timestamp columns describe the arrival/version boundary and should
    # not force a new version by themselves.
    hash_columns = [
        c for c in frame.columns
        if c not in ignore_columns_to_hash
        and c not in {DPM_VERSION_COLUMN, DPM_INGESTION_COLUMN}
    ]

    source_h = frame.withColumn("__src_hash", _payload_hash(frame, hash_columns))

    current_payload = (
        target
        .filter(F.col("ctrl_state") == F.lit(1))
        .select(
            *[F.col(c) for c in frame.columns],
            *[F.col(c) for c in join_keys if c not in frame.columns],
        )
    )
    current_h = current_payload.withColumn(
        "__tgt_hash",
        _payload_hash(current_payload, hash_columns),
    )

    joined = (
        source_h.alias("s")
        .join(
            current_h.alias("t"),
            on=[
                F.col(f"s.{key}").eqNullSafe(F.col(f"t.{key}"))
                for key in join_keys
            ],
            how="left",
        )
    )

    target_exists_marker = F.col(f"t.{join_keys[0]}").isNotNull()

    # Keys can legitimately contain nulls, so use the target hash as the match marker.
    is_match = F.col("t.__tgt_hash").isNotNull()
    is_changed = (~is_match) | (
        F.col("s.__src_hash") != F.col("t.__tgt_hash")
    )

    source_columns = [
        F.col(f"s.{c}").alias(c)
        for c in frame.columns
    ]

    changed_or_new = (
        joined
        .filter(is_changed)
        .select(*source_columns)
        .cache()
    )

    unchanged = (
        joined
        .filter(is_match & ~is_changed)
        .select(*source_columns)
        .cache()
    )

    changed_count = changed_or_new.count()
    unchanged_count = unchanged.count()

    delta = DeltaTable.forName(spark, target_table)
    merge_condition = _quoted_join_condition("t", "s", join_keys) + " AND t.ctrl_state = 1"

    # Close the current active version for changed existing keys.
    if changed_count:
        changed_keys = changed_or_new.select(
            *join_keys,
            F.col(DPM_VERSION_COLUMN),
            F.col(DPM_INGESTION_COLUMN),
        )

        (
            delta.alias("t")
            .merge(changed_keys.alias("s"), merge_condition)
            .whenMatchedUpdate(
                set={
                    "ref_end_dt": f"s.`{DPM_VERSION_COLUMN}`",
                    "ctrl_end_dt": f"s.`{DPM_INGESTION_COLUMN}`",
                    "ctrl_updt_dt": f"s.`{DPM_INGESTION_COLUMN}`",
                    "ctrl_state": "0",
                }
            )
            .execute()
        )

    # Rows seen again without a business change remain the same SCD version.
    # We only advance "last seen"/control timestamps.
    if unchanged_count:
        unchanged_keys = unchanged.select(
            *join_keys,
            F.col(DPM_VERSION_COLUMN),
            F.col(DPM_INGESTION_COLUMN),
        )

        (
            delta.alias("t")
            .merge(unchanged_keys.alias("s"), merge_condition)
            .whenMatchedUpdate(
                set={
                    "ref_last_dt": f"s.`{DPM_VERSION_COLUMN}`",
                    "ctrl_updt_dt": f"s.`{DPM_INGESTION_COLUMN}`",
                }
            )
            .execute()
        )

    # Allocate surrogate keys only after closing old versions.
    if changed_count:
        max_sk = (
            spark.table(target_table)
            .agg(F.max(F.col(sk_column)).alias("max_sk"))
            .first()["max_sk"]
        )
        max_sk = int(max_sk or 0)

        new_versions = _with_new_scd_control_columns(
            changed_or_new,
            table_name=target_table,
            base_sk=max_sk,
        )

        (
            new_versions.select(*spark.table(target_table).columns)
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

    changed_or_new.unpersist()
    unchanged.unpersist()

    return {
        "created": False,
        "inserted_versions": changed_count,
        "closed_versions": changed_count,
        "unchanged_rows": unchanged_count,
    }


StatementMeta(, c5e6a7b0-a131-4635-85b4-3f25991aaad8, 6, Finished, Available, Finished, False)

## 3. Load-specific preparation


In [4]:
import re
import uuid
from pathlib import Path

import pandas as pd
from pyspark.sql import types as T

SHEET_TO_TABLE = {
    "Datapoints": "dp_lgl_reg_dpm_param_datapoint",
    "Filters": "dp_lgl_reg_dpm_param_fltr",
    "Catalog": "dp_lgl_reg_dpm_param_ctlg",
    # One row per source table used by the file.
    "Sources": "dp_lgl_reg_dpm_param_src_tbl",
}

# Excel header -> physical column (DDA naming convention). A header not listed here is
# loaded with its name in snake_case.
COLUMN_RENAMES = {
    "Datapoints": {
        "Framework": "framework_cd",
        "ID Datapoint": "datapoint_id",
        "Variable (v#)": "var_cd",
        "Description (Auto)": "datapoint_desc",
        "Type": "datapoint_cl_typ",
        "Source Table": "src_tbl_id",
        "Metric / Formula": "metric_fmla_ln",
        "Having (aggregate condition)": "having_ln",
        "Period": "per_expr_cd",
        "Check": "excel_check_ln",
    },
    "Filters": {
        "ID Datapoint": "datapoint_id",
        "Variable (v#)": "var_cd",
        "Column": "src_column_nm",
        "Operator": "optr_cd",
        "Values": "fltr_val_ln",
        "Check": "excel_check_ln",
    },
    "Catalog": {
        "ID Datapoint": "datapoint_id",
        "Framework": "framework_cd",
        "Annex": "anx_cd",
        "Template": "tmplt_cd",
        "Row": "row_cd",
        "Column": "column_cd",
        "Description": "datapoint_desc",
        "Row Description": "row_desc",
        "Column Level 1": "column_lvl_1_desc",
        "Column Level 2": "column_lvl_2_desc",
        "Value Type": "val_typ",
        "Check": "excel_check_ln",
    },
    # Role columns: physical column that plays the role of entity, IDENT, sub-entity and
    # country; empty = the role is not part of the grouping key.
    "Sources": {
        "Source Table": "src_tbl_id",
        "Physical Table": "phy_tbl_nm",
        "Period Column": "per_column_nm",
        "Period Format": "per_fmt_cd",
        "Entity Column": "ent_column_nm",
        "Ident Column": "ident_column_nm",
        "Subentity Column": "subent_column_nm",
        "Country Column": "ctry_column_nm",
        "Grain ID": "grain_id",
    },
}

# Sources tab: fixed columns, in this order. The first four headers are mandatory; the
# role columns and Grain ID may be missing and are loaded empty. Other headers are ignored.
SOURCES_COLUMNS = list(COLUMN_RENAMES["Sources"].values())
SOURCES_REQUIRED_HEADERS = ["Source Table", "Physical Table", "Period Column", "Period Format"]

METADATA_JOIN_KEYS = ["src_file_nm", "src_she_nm", "src_row_nr"]
METADATA_IGNORE_COLUMNS_TO_HASH = ["metadata_load_id", "load_dt"]



def local_path(path):
    value = str(path).strip()
    if value.startswith("file:"):
        value = value[5:]
    return Path(value)


def list_xlsx_files(folder):
    p = local_path(folder)
    if not p.exists():
        raise FileNotFoundError(
            f"Metadata folder not found: {p}. "
            "Attach the target Lakehouse as the DEFAULT Lakehouse."
        )
    if not p.is_dir():
        raise NotADirectoryError(f"Metadata path is not a directory: {p}")

    return sorted(
        str(x)
        for x in p.iterdir()
        if x.is_file()
        and x.suffix.lower() == ".xlsx"
        and not x.name.startswith("~$")
    )


def generic_column_name(name):
    text = str(name).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text or "unnamed"


def normalize_sheet_columns(sheet_name, frame):
    explicit = COLUMN_RENAMES[sheet_name]
    renamed = {}
    used = set()

    for col in frame.columns:
        target = explicit.get(str(col), generic_column_name(col))
        if target in used:
            raise ValueError(
                f"Duplicate normalized column {target!r} in sheet {sheet_name!r}."
            )
        used.add(target)
        renamed[col] = target

    return frame.rename(columns=renamed)


def read_workbook(path):
    p = local_path(path)
    source_file = p.name

    excel = pd.ExcelFile(p)
    missing = sorted(set(required_sheets) - set(excel.sheet_names))
    if missing:
        raise ValueError(
            f"Workbook {source_file} is missing required sheets: {missing}"
        )

    output = {}
    for sheet in required_sheets:
        pdf = pd.read_excel(
            p,
            sheet_name=sheet,
            dtype=str,
            keep_default_na=False,
        )
        if sheet == "Sources":
            missing_headers = [h for h in SOURCES_REQUIRED_HEADERS if h not in pdf.columns]
            if missing_headers:
                raise ValueError(
                    f"Workbook {source_file}: sheet Sources is missing column(s) {missing_headers}"
                )
            ignored = [str(c) for c in pdf.columns if str(c) not in COLUMN_RENAMES["Sources"]]
            if ignored:
                print(f"{source_file} / Sources: columns not loaded {ignored}")
        pdf = normalize_sheet_columns(sheet, pdf).fillna("")

        for col in pdf.columns:
            pdf[col] = pdf[col].astype(str).map(lambda value: value.strip())

        if sheet == "Sources":
            for column in SOURCES_COLUMNS:
                if column not in pdf.columns:
                    pdf[column] = ""
            pdf = pdf[SOURCES_COLUMNS]

        # The physical row identity is independent of business validity: invalid or duplicate
        # rows must still be loaded so that the validation can report them.
        pdf["src_file_nm"] = source_file
        pdf["src_she_nm"] = sheet
        pdf["src_row_nr"] = range(2, len(pdf) + 2)
        if sheet == "Sources":
            # Blank rows at the end of the tab are not source tables.
            pdf = pdf[pdf[SOURCES_COLUMNS].ne("").any(axis=1)]
        output[sheet] = pdf

    return output


def metadata_spark_frame(pdf, metadata_load_id, load_timestamp, target_table):
    if pdf.empty:
        # An empty Filters/Catalog/Sources slice is a valid input: keep the declared columns so
        # that SCD 2.2 can close the rows that disappeared from the full snapshot.
        existing_types = {}
        if spark.catalog.tableExists(target_table):
            existing_types = {
                field.name: field.dataType
                for field in spark.table(target_table).schema.fields
            }
        schema = T.StructType([
            T.StructField(
                str(column),
                existing_types.get(str(column), T.StringType()),
                True,
            )
            for column in pdf.columns
        ])
        frame = spark.createDataFrame([], schema)
    else:
        frame = spark.createDataFrame(pdf)

    return (
        frame
        .withColumn("metadata_load_id", F.lit(metadata_load_id))
        .withColumn("load_dt", F.lit(load_timestamp).cast("timestamp"))
        .withColumn(DPM_PRESENT_COLUMN, F.lit(1).cast("int"))
        .withColumn(DPM_VERSION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
        .withColumn(DPM_INGESTION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
    )


# Excel header of the issue catalogue -> physical column (DDA naming convention).
ISSUE_CATALOG_COLUMNS = {
    "issue_code": "iss_cd",
    "domain": "dmn_cd",
    "stage": "stg_cd",
    "pattern_code": "pattern_cd",
    "severity": "sev_cd",
    "scope": "scp_cd",
    "message_template": "msg_tmplt_ln",
    "param_names": "param_nm_list_ln",
    "active": "act_flg",
}
ISSUE_CATALOG_REQUIRED_COLUMNS = list(ISSUE_CATALOG_COLUMNS)
ISSUE_CATALOG_ACTIVE_VALUES = {"1", "TRUE", "T", "Y", "YES", "S", "SI"}


def read_issue_catalog(path):
    p = local_path(path)
    if not p.exists():
        raise FileNotFoundError(f"Issue catalog workbook not found: {p}")

    pdf = pd.read_excel(
        p,
        sheet_name=issue_catalog_sheet,
        dtype=str,
        keep_default_na=False,
    ).fillna("")
    pdf = pdf.rename(columns={c: generic_column_name(c) for c in pdf.columns})
    for col in pdf.columns:
        pdf[col] = pdf[col].astype(str).map(lambda value: value.strip())

    missing = sorted(set(ISSUE_CATALOG_REQUIRED_COLUMNS) - set(pdf.columns))
    if missing:
        raise ValueError(
            f"Issue catalog is missing required column(s): {missing}"
        )
    if pdf.empty:
        raise ValueError("Issue catalog is empty.")
    duplicates = pdf[pdf["issue_code"].duplicated(keep=False)]["issue_code"].tolist()
    if duplicates:
        raise ValueError(
            f"Issue catalog contains duplicate issue_code values: {sorted(set(duplicates))}"
        )
    pdf = pdf[ISSUE_CATALOG_REQUIRED_COLUMNS].rename(columns=ISSUE_CATALOG_COLUMNS)
    # _flg columns are INT 0/1.
    pdf["act_flg"] = pdf["act_flg"].map(
        lambda value: 1 if str(value).strip().upper() in ISSUE_CATALOG_ACTIVE_VALUES else 0
    )
    return pdf


def issue_catalog_spark_frame(pdf, load_timestamp):
    text_columns = [c for c in pdf.columns if c != "act_flg"]
    frame = spark.createDataFrame(
        pdf.astype({c: str for c in text_columns}),
        T.StructType(
            [T.StructField(c, T.StringType(), True) for c in text_columns]
            + [T.StructField("act_flg", T.IntegerType(), True)]
        ) if list(pdf.columns) == text_columns + ["act_flg"] else None,
    )
    return (
        frame
        .withColumn(DPM_PRESENT_COLUMN, F.lit(1).cast("int"))
        .withColumn(DPM_VERSION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
        .withColumn(DPM_INGESTION_COLUMN, F.lit(load_timestamp).cast("timestamp"))
    )


StatementMeta(, c5e6a7b0-a131-4635-85b4-3f25991aaad8, 7, Finished, Available, Finished, False)

## 4. Execute SCD 2.2 load


In [5]:
# One version boundary is shared by Datapoints, Filters, Catalog and Sources.
metadata_version_timestamp = resolve_load_timestamp(load_version_timestamp)
metadata_load_id = str(uuid.uuid4())

xlsx_files = list_xlsx_files(metadata_input_path)
if fail_on_empty_folder and not xlsx_files:
    raise ValueError(f"No .xlsx metadata files found in {metadata_input_path}")

print(f"Metadata files       : {len(xlsx_files)}")
print(f"Metadata version UTC : {metadata_version_timestamp.isoformat()}")
print(f"Metadata load id     : {metadata_load_id}")

# Read and validate every workbook before the first table is published.
per_sheet = {sheet: [] for sheet in required_sheets}
for path in xlsx_files:
    workbook = read_workbook(path)
    for sheet, pdf in workbook.items():
        per_sheet[sheet].append(pdf)

prepared = {}
for sheet in required_sheets:
    combined = (
        pd.concat(per_sheet[sheet], ignore_index=True, sort=False)
        if per_sheet[sheet]
        else pd.DataFrame()
    ).fillna("")

    target_table = SHEET_TO_TABLE[sheet]
    frame = metadata_spark_frame(
        combined,
        metadata_load_id=metadata_load_id,
        load_timestamp=metadata_version_timestamp,
        target_table=target_table,
    )

    frame, tombstone_count = build_full_snapshot_with_tombstones(
        target_table,
        frame,
        METADATA_JOIN_KEYS,
        metadata_version_timestamp,
        snapshot_mode="FULL",
    )
    prepared[sheet] = (target_table, frame, tombstone_count)


if bool(load_issue_catalog):
    issue_pdf = read_issue_catalog(issue_catalog_input_path)
    issue_frame = issue_catalog_spark_frame(
        issue_pdf,
        load_timestamp=metadata_version_timestamp,
    )
    issue_frame, issue_tombstones = build_full_snapshot_with_tombstones(
        issue_catalog_table,
        issue_frame,
        ["iss_cd"],
        metadata_version_timestamp,
        snapshot_mode="FULL",
    )
    prepared[issue_catalog_sheet] = (
        issue_catalog_table,
        issue_frame,
        issue_tombstones,
    )

# Publish only after all prepared snapshots have passed the local preflight.
summary = []
publish_items = list(required_sheets) + (
    [issue_catalog_sheet] if bool(load_issue_catalog) else []
)

for sheet in publish_items:
    target_table, frame, tombstone_count = prepared[sheet]
    join_keys = (
        ["iss_cd"]
        if sheet == issue_catalog_sheet
        else METADATA_JOIN_KEYS
    )
    ignore_columns_to_hash = (
        []
        if sheet == issue_catalog_sheet
        else METADATA_IGNORE_COLUMNS_TO_HASH
    )

    result = spark_scd22_upsert(
        frame=frame,
        target_table=target_table,
        join_keys=join_keys,
        ignore_columns_to_hash=ignore_columns_to_hash,
    )

    summary.append((target_table, tombstone_count, result))

    print(
        f"Published {target_table} directly with Spark/Delta SCD 2.2 "
        f"(created={result['created']}, "
        f"inserted_versions={result['inserted_versions']}, "
        f"closed_versions={result['closed_versions']}, "
        f"unchanged_rows={result['unchanged_rows']}, "
        f"tombstones={tombstone_count})."
    )

print("")
print("Metadata SCD load completed.")
print(f"Version timestamp: {metadata_version_timestamp.isoformat()}")
for table_name, tombstone_count, result in summary:
    print(
        f" - {table_name}: "
        f"inserted={result['inserted_versions']}, "
        f"closed={result['closed_versions']}, "
        f"unchanged={result['unchanged_rows']}, "
        f"tombstones={tombstone_count}"
    )


StatementMeta(, c5e6a7b0-a131-4635-85b4-3f25991aaad8, 8, Finished, Available, Finished, False)

Metadata files       : 4
Metadata version UTC : 2026-11-20T09:25:13.095877
Metadata load id     : d1923d09-7206-4d9d-b422-d55ea2fa38f7


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


ValueError: Incoming snapshot for 'dp_lgl_reg_dpm_param_datapoint' contains duplicate SCD join keys ['src_file_nm', 'src_she_nm', 'src_row_nr']. SCD Type 2.2 requires one incoming row per logical key.

## Snapshot contract used by Validation and Processing

This notebook now implements the SCD Type 2.2 publication directly with Spark/Delta.

- `dpm_vrs_dt` is the functional version boundary.
- `dpm_ingestion_dt` is the ingestion/control timestamp.
- Active rows have `ctrl_state = 1`.
- Closed historical rows have `ctrl_state = 0` and a populated `ref_end_dt`.
- `ref_start_dt` marks the beginning of a version.
- `ref_last_dt` is advanced when the same logical row is seen again unchanged.
- `row_present_flg` preserves full-snapshot delete semantics. A row removed from
  a later FULL snapshot receives a tombstone version with `row_present_flg = 0`.
- `metadata_load_id` and `load_dt` are ignored when deciding whether a
  business change requires a new SCD version.
- `dpm_vrs_dt` and `dpm_ingestion_dt` also do not trigger a new version by
  themselves.

Historical reads continue to use:

```text
ref_start_dt <= as_of
AND (ref_end_dt > as_of OR ref_end_dt IS NULL)
AND row_present_flg = 1
```

### First load

If a target table does not exist, the notebook creates the Delta table directly
from the prepared snapshot and adds the SCD control columns.

### Existing tables

An existing target must already contain the SCD control columns and the same
payload columns as the incoming snapshot. Schema drift fails fast rather than
silently altering historical tables.
